In [ ]:
import os
import time

import psycopg2
import numpy as np
import pandas as pd
from qdrant_client import QdrantClient
from pgvector.psycopg2 import register_vector
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue, Range
)

In [19]:
### If using Docker, these environment variables are already set
db_host = os.getenv('POSTGRES_HOST', 'localhost')
conn = psycopg2.connect(
    host=db_host,
    database="arxiv_db",
    user="postgres",
    password="lesson_password"
)
cur = conn.cursor()

In [ ]:


### Enable pgvector extension
### This needs to happen BEFORE we register the vector type
cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
conn.commit()

In [21]:
papers_df = pd.read_csv('../arxiv_papers_5k.csv')
embeddings = np.load('../embeddings_cohere_5k.npy')

In [22]:
papers_df.head()

,title,abstract,authors,published,category,arxiv_id
0,Optimizing Mixture of Block Attention,"Mixture of Block Attention (MoBA) (Lu et al., ...","Guangxuan Xiao, Junxian Guo, Kasra Mazaheri, S...",2025-11-14 18:59:59+00:00,cs.LG,2511.11571v1
1,Estimating Total Effects in Bipartite Experime...,We study randomized experiments in bipartite s...,"Albert Tan, Mohsen Bayati, James Nordlund, Rom...",2025-11-14 18:55:51+00:00,cs.LG,2511.11564v1
2,A Unified Convergence Analysis for Semi-Decent...,"In semi-decentralized federated learning, devi...","Angelo Rodio, Giovanni Neglia, Zheng Chen, Eri...",2025-11-14 18:53:37+00:00,cs.LG,2511.11560v1
3,Multistability of Self-Attention Dynamics in T...,"In machine learning, a self-attention dynamics...",Claudio Altafini,2025-11-14 18:45:22+00:00,cs.LG,2511.11553v1
4,Generalizing Fair Clustering to Multiple Group...,Clustering is a fundamental task in machine le...,"Diptarka Chakraborty, Kushagra Chatterjee, Deb...",2025-11-14 18:19:18+00:00,cs.LG,2511.11539v1


In [23]:
### register the vector type with psycopg2
### This lets us pass numpy arrays directly as vectors
register_vector(conn)


In [24]:

### Create table for our papers
### The vector(1536) column stores our 1536-dimensional embeddings
cur.execute("""
    CREATE TABLE IF NOT EXISTS papers (
        id TEXT PRIMARY KEY,
        arxiv_id TEXT,
        title TEXT,
        authors TEXT,
        abstract TEXT,
        year INTEGER,
        category TEXT,
        embedding vector(1536)
    )
""")
conn.commit()

In [25]:





### Load the metadata and embeddings


print(f"Loading {len(papers_df)} papers into PostgreSQL...")

### Insert papers in batches
### We'll do 500 at a time to keep transactions manageable
batch_size = 500
for i in range(0, len(papers_df), batch_size):
    batch_df = papers_df.iloc[i:i+batch_size]
    batch_embeddings = embeddings[i:i+batch_size]

    for j, (idx, row) in enumerate(batch_df.iterrows()):
        cur.execute("""
            INSERT INTO papers (id, title, authors, abstract, year, category, embedding)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (id) DO NOTHING
        """, (
            row['arxiv_id'],
            row['title'],
            row['authors'],
            row['abstract'],
            row['published'].split(" ")[0].replace("-", ""),
            row['category'],
            batch_embeddings[j]  # Pass numpy array directly
        ))

    conn.commit()
    print(f"  Loaded {min(i+batch_size, len(papers_df))} / {len(papers_df)} papers")

print("\nData loaded successfully!")

### Create HNSW index for fast similarity search
### This takes a couple seconds for 5,000 papers
print("Creating HNSW index...")
cur.execute("""
    CREATE INDEX IF NOT EXISTS papers_embedding_idx
    ON papers
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64)
""")
conn.commit()
print("Index created!")

### Verify everything worked
cur.execute("SELECT COUNT(*) FROM papers")
count = cur.fetchone()[0]
print(f"\nTotal papers in database: {count}")


Loading 5000 papers into PostgreSQL...
  Loaded 500 / 5000 papers
  Loaded 1000 / 5000 papers
  Loaded 1500 / 5000 papers
  Loaded 2000 / 5000 papers
  Loaded 2500 / 5000 papers
  Loaded 3000 / 5000 papers
  Loaded 3500 / 5000 papers
  Loaded 4000 / 5000 papers
  Loaded 4500 / 5000 papers
  Loaded 5000 / 5000 papers

Data loaded successfully!
Creating HNSW index...
Index created!

Total papers in database: 4708


In [5]:
cur.execute("""
    SELECT id, title, category, year, embedding
    FROM papers
    WHERE category = 'cs.LG'
    LIMIT 1
""")
result = cur.fetchone()
query_id, query_title, query_category, query_year, query_embedding = result


In [6]:
### Scenario 1: Unfiltered similarity search
### The <=> operator computes cosine distance
print("=" * 80)
print("Scenario 1: Unfiltered Similarity Search")
print("=" * 80)
cur.execute("""
    SELECT title, category, year, embedding <=> %s AS distance
    FROM papers
    WHERE id != %s
    ORDER BY embedding <=> %s
    LIMIT 5
""", (query_embedding, query_id, query_embedding))

for row in cur.fetchall():
    print(f"  {row[1]:8} {row[2]} | {row[3]:.4f} | {row[0][:60]}")
print()

Scenario 1: Unfiltered Similarity Search
  cs.LG    20251108 | 0.4332 | MoSKA: Mixture of Shared KV Attention for Efficient Long-Seq
  cs.LG    20251112 | 0.4615 | Mixture-of-Channels: Exploiting Sparse FFNs for Efficient LL
  cs.CL    20251110 | 0.5123 | Pre-Attention Expert Prediction and Prefetching for Mixture-
  cs.CL    20251102 | 0.5342 | Optimizing Native Sparse Attention with Latent Attention and
  cs.CL    20251101 | 0.5403 | FlashEVA: Accelerating LLM inference via Efficient Attention



In [7]:
### Scenario 2: Filter by category
print("=" * 80)
print("Scenario 2: Category Filter (cs.LG only)")
print("=" * 80)
cur.execute("""
    SELECT title, category, year, embedding <=> %s AS distance
    FROM papers
    WHERE category = 'cs.LG' AND id != %s
    ORDER BY embedding <=> %s
    LIMIT 5
""", (query_embedding, query_id, query_embedding))

for row in cur.fetchall():
    print(f"  {row[1]:8} {row[2]} | {row[3]:.4f} | {row[0][:60]}")

Scenario 2: Category Filter (cs.LG only)
  cs.LG    20251108 | 0.4332 | MoSKA: Mixture of Shared KV Attention for Efficient Long-Seq
  cs.LG    20251112 | 0.4615 | Mixture-of-Channels: Exploiting Sparse FFNs for Efficient LL
  cs.LG    20251112 | 0.5465 | Making Every Head Count: Sparse Attention Without the Speed-
  cs.LG    20251107 | 0.5477 | Attention and Compression is all you need for Controllably E
  cs.LG    20251109 | 0.5589 | Route Experts by Sequence, not by Token


In [30]:
### Scenario 3: Filter by year range
print("=" * 80)
print("Scenario 3: Year Filter (2025 or later)")
print("=" * 80)
cur.execute("""
    SELECT title, category, year, embedding <=> %s AS distance
    FROM papers
    WHERE year >= 2025 AND id != %s
    ORDER BY embedding <=> %s DESC
    LIMIT 5
""", (query_embedding, query_id, query_embedding))

for row in cur.fetchall():
    print(f"  {row[1]:8} {row[2]} | {row[3]:.4f} | {row[0][:60]}")
print()

Scenario 3: Year Filter (2025 or later)
  cs.SE    20251022 | 0.9900 | LifeSync-Games: Toward a Video Game Paradigm for Promoting R
  cs.SE    20251108 | 0.9618 | Digital Twins and Their Applications in Modeling Different L
  cs.DB    20250702 | 0.9567 | A bibliometric analysis on the current situation and hot tre
  cs.SE    20251007 | 0.9563 | Digital Twins for Software Engineering Processes
  cs.SE    20251108 | 0.9553 | The Impact of COVID-19 and Remote Work on Software Developme



In [9]:
### Scenario 4: Combined filters
print("=" * 80)
print("Scenario 4: Combined Filter (cs.LG AND year >= 2025)")
print("=" * 80)
cur.execute("""
    SELECT title, category, year, embedding <=> %s AS distance
    FROM papers
    WHERE category = 'cs.LG' AND year >= 2025 AND id != %s
    ORDER BY embedding <=> %s DESC
    LIMIT 5
""", (query_embedding, query_id, query_embedding))

for row in cur.fetchall():
    print(f"  {row[1]:8} {row[2]} | {row[3]:.4f} | {row[0][:60]}")

Scenario 4: Combined Filter (cs.LG AND year >= 2025)
  cs.LG    20251113 | 0.9429 | Gradient Flow Equations for Deep Linear Neural Networks: A S
  cs.LG    20251107 | 0.9415 | Primal-Only Actor Critic Algorithm for Robust Constrained Av
  cs.LG    20251110 | 0.9338 | Neyman-Pearson Classification under Both Null and Alternativ
  cs.LG    20251113 | 0.9336 | Robot Crash Course: Learning Soft and Stylized Falling
  cs.LG    20251109 | 0.9133 | Bridging Theory and Practice: A Stochastic Learning-Optimiza


In [10]:
cur.execute("""
    SELECT embedding FROM papers
    WHERE category = 'cs.LG'
    LIMIT 1
""")
query_embedding = cur.fetchone()[0]

In [16]:
def benchmark_query(query, params, name, iterations=100):
    """Run a query multiple times and measure average latency"""
    # Warmup
    for _ in range(5):
        cur.execute(query, params)
        cur.fetchall()

    # Actual measurement
    times = []
    for _ in range(iterations):
        start = time.time()
        cur.execute(query, params)
        cur.fetchall()
        times.append((time.time() - start) * 1000)  # Convert to ms

    avg_time = np.mean(times)
    std_time = np.std(times)
    return avg_time, std_time

In [26]:
print("Benchmarking pgvector performance...")
print("=" * 80)

### Scenario 1: Unfiltered
query1 = """
    SELECT title, category, year, embedding <=> %s AS distance
    FROM papers
    ORDER BY embedding <=> %s
    LIMIT 10
"""
avg, std = benchmark_query(query1, (query_embedding, query_embedding), "Unfiltered")
print(f"Unfiltered search:        {avg:.2f}ms (±{std:.2f}ms)")
baseline = avg

### Scenario 2: Category filter
query2 = """
    SELECT title, category, year, embedding <=> %s AS distance
    FROM papers
    WHERE category = 'cs.LG'
    ORDER BY embedding <=> %s
    LIMIT 10
"""
avg, std = benchmark_query(query2, (query_embedding, query_embedding), "Category filter")
overhead = avg / baseline
print(f"Category filter:          {avg:.2f}ms (±{std:.2f}ms) | {overhead:.2f}x overhead")

### Scenario 3: Year filter
query3 = """
    SELECT title, category, year, embedding <=> %s AS distance
    FROM papers
    WHERE year >= 20250929
    ORDER BY embedding <=> %s
    LIMIT 10
"""
avg, std = benchmark_query(query3, (query_embedding, query_embedding), "Year filter")
overhead = avg / baseline
print(f"Year filter (>= 20250929):    {avg:.2f}ms (±{std:.2f}ms) | {overhead:.2f}x overhead")

### Scenario 4: Combined filters
query4 = """
    SELECT title, category, year, embedding <=> %s AS distance
    FROM papers
    WHERE category = 'cs.LG' AND year >= 20250929
    ORDER BY embedding <=> %s
    LIMIT 10
"""
avg, std = benchmark_query(query4, (query_embedding, query_embedding), "Combined filter")
overhead = avg / baseline
print(f"Combined filter:          {avg:.2f}ms (±{std:.2f}ms) | {overhead:.2f}x overhead")

print("=" * 80)

Benchmarking pgvector performance...
Unfiltered search:        25.26ms (±4.95ms)
Category filter:          9.71ms (±2.05ms) | 0.38x overhead
Year filter (>= 20250929):    20.78ms (±3.11ms) | 0.82x overhead
Combined filter:          9.96ms (±2.37ms) | 0.39x overhead


In [18]:
cur.close()
conn.close()

In [ ]:
"""
What pgvector Doesn’t Handle;
- VACUUM is Your Problem: PostgreSQL’s VACUUM process can interact weirdly
  with vector indexes. The indexes can bloat over time if one's doing lots
  of updates and deletes. You need to monitor this and potentially
  rebuild indexes periodically.

- Index Maintenance: As ones data grows and changes, you might need to
  rebuild indexes to maintain performance. This isn’t automatic.
  It’s part of ones operational responsibility.

- Memory Pressure: Vector indexes live in memory for best performance.
  As your dataset grows, you need to size your database appropriately.
  This is normal for PostgreSQL, but it’s something you have to plan for.

- Replication Overhead: If one's replicating your PostgreSQL database,
  those vector columns come along for the ride. Replicating high-dimensional
  vectors can be bandwidth-intensive.
"""

In [ ]:
"""
pgvector is an excellent choice when:
    - already run PostgreSQL in production
    - team has strong SQL skills and PostgreSQL experience
    - need transactional guarantees with vector operations
    - primarily filter on integer fields (dates, IDs, counts, years)
    - scale is moderate (up to a few million vectors)
    - want to leverage existing PostgreSQL infrastructure

pgvector might not be the best fit when:

    - filtering heavily on text fields with unpredictable combinations
    - there's a need to scale beyond what a single PostgreSQL server can handle
    - ones team doesn't have PostgreSQL operational expertise
    - someone else is to handle all the database maintenance
"""

/Users/sbamwite/Desktop/rags/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [29]:
qdrant_host = os.getenv('QDRANT_HOST', 'localhost')
client = QdrantClient(host=qdrant_host, port=6333)

In [30]:
collection_name = "arxiv_papers"

### Delete collection if it exists (useful for re-running)
try:
    client.delete_collection(collection_name)
    print(f"Deleted existing collection: {collection_name}")
except:
    print("no collections to delete")

Deleted existing collection: arxiv_papers


In [31]:
### Create new collection
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=1536,  # Cohere embedding dimension
        distance=Distance.COSINE
    )
)
print(f"Created collection: {collection_name}")

Created collection: arxiv_papers


In [34]:
print(f"\nLoading {len(papers_df)} papers into Qdrant...")

### Prepare points for upload
### Qdrant stores metadata as "payload"
points = []
for idx, row in papers_df.iterrows():
    point = PointStruct(
        id=idx,
        vector=embeddings[idx].tolist(),
        payload={
            "paper_id": row['arxiv_id'],
            "title": row['title'],
            "authors": row['authors'],
            "abstract": row['abstract'],
            "year": row['published'].split(" ")[0].replace("-", ""),
            "category": row['category']
        }
    )
    points.append(point)

    # Show progress every 1000 papers
    if (idx + 1) % 1000 == 0:
        print(f"  Prepared {idx + 1} / {len(papers_df)} papers")



Loading 5000 papers into Qdrant...
  Prepared 1000 / 5000 papers
  Prepared 2000 / 5000 papers
  Prepared 3000 / 5000 papers
  Prepared 4000 / 5000 papers
  Prepared 5000 / 5000 papers


In [35]:
### Upload all points at once
### Qdrant handles large batches well (no 5k limit like ChromaDB)
print("\nUploading to Qdrant...")
client.upsert(
    collection_name=collection_name,
    points=points
)

print(f"Upload complete!")


Uploading to Qdrant...


UnexpectedResponse: Unexpected Response: 400 (Bad Request)
Raw response content:
b'{"status":{"error":"Payload error: JSON payload (104259560 bytes) is larger than allowed (limit: 33554432 bytes)."},"time":0.0}'

In [39]:
# allows upload in batches coz of below error;
"""
b'{"status":{"error":"Payload error: JSON payload (104259560 bytes) is larger than allowed (limit: 33554432 bytes)."},"time":0.0}'
"""
for i in range(0, len(points), batch_size):
    batch = points[i:i+batch_size]
    client.upsert(collection_name=collection_name, points=batch)
    print(f"Uploaded {i + len(batch)} out of {len(points)} points")

Uploaded 500 out of 5000 points
Uploaded 1000 out of 5000 points
Uploaded 1500 out of 5000 points
Uploaded 2000 out of 5000 points
Uploaded 2500 out of 5000 points
Uploaded 3000 out of 5000 points
Uploaded 3500 out of 5000 points
Uploaded 4000 out of 5000 points
Uploaded 4500 out of 5000 points
Uploaded 5000 out of 5000 points


In [40]:
### Verify
collection_info = client.get_collection(collection_name)
print(f"\nCollection '{collection_name}' now has {collection_info.points_count} papers")


Collection 'arxiv_papers' now has 5000 papers


In [42]:
### Get a query vector from a machine learning paper
results = client.scroll(
    collection_name=collection_name,
    scroll_filter=Filter(
        must=[FieldCondition(key="category", match=MatchValue(value="cs.LG"))]
    ),
    limit=1,
    with_vectors=True,
    with_payload=True
)

In [44]:
query_point = results[0][0]
query_vector = query_point.vector
query_payload = query_point.payload

print("Query paper:")
print(f"  Title: {query_payload['title']}")
print(f"  Category: {query_payload['category']}")
print(f"  Year: {query_payload['year']}")
print()

### Scenario 1: Unfiltered similarity search
print("=" * 80)
print("Scenario 1: Unfiltered Similarity Search")
print("=" * 80)

results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=6,  # Get 6 so we can skip the query paper itself
    with_payload=True
)

for hit in results.points[1:6]:  # Skip first result (the query paper)
    payload = hit.payload
    print(f"  {payload['category']:8} {payload['year']} | {hit.score:.4f} | {payload['title'][:60]}")
print()

Query paper:
  Title: Optimizing Mixture of Block Attention
  Category: cs.LG
  Year: 20251114

Scenario 1: Unfiltered Similarity Search
  cs.CL    20251114 | 1.0000 | Optimizing Mixture of Block Attention
  cs.LG    20251108 | 0.5668 | MoSKA: Mixture of Shared KV Attention for Efficient Long-Seq
  cs.LG    20251112 | 0.5385 | Mixture-of-Channels: Exploiting Sparse FFNs for Efficient LL
  cs.CL    20251110 | 0.4877 | Pre-Attention Expert Prediction and Prefetching for Mixture-
  cs.CL    20251102 | 0.4658 | Optimizing Native Sparse Attention with Latent Attention and



In [45]:
### Scenario 2: Filter by category
print("=" * 80)
print("Scenario 2: Category Filter (cs.LG only)")
print("=" * 80)

results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    query_filter=Filter(
        must=[FieldCondition(key="category", match=MatchValue(value="cs.LG"))]
    ),
    limit=6,
    with_payload=True
)

for hit in results.points[1:6]:
    payload = hit.payload
    print(f"  {payload['category']:8} {payload['year']} | {hit.score:.4f} | {payload['title'][:60]}")
print()

Scenario 2: Category Filter (cs.LG only)
  cs.LG    20251108 | 0.5668 | MoSKA: Mixture of Shared KV Attention for Efficient Long-Seq
  cs.LG    20251112 | 0.5385 | Mixture-of-Channels: Exploiting Sparse FFNs for Efficient LL
  cs.LG    20251112 | 0.4535 | Making Every Head Count: Sparse Attention Without the Speed-
  cs.LG    20251107 | 0.4523 | Attention and Compression is all you need for Controllably E
  cs.LG    20251109 | 0.4411 | Route Experts by Sequence, not by Token



In [52]:
### Scenario 3: Filter by year range
print("=" * 80)
print("Scenario 3: Year Filter (2025 or later)")
print("=" * 80)

results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    query_filter=Filter(
        must=[FieldCondition(key="year", match=MatchValue(value='20251129'))]
    ),
    limit=5,
    with_payload=True
)

for hit in results.points:
    payload = hit.payload
    print(f"  {payload['category']:8} {payload['year']} | {hit.score:.4f} | {payload['title'][:60]}")
print()

Scenario 3: Year Filter (2025 or later)



In [ ]:
### Scenario 4: Combined filters
print("=" * 80)
print("Scenario 4: Combined Filter (cs.LG AND year >= 2025)")
print("=" * 80)

results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    query_filter=Filter(
        must=[
            FieldCondition(key="category", match=MatchValue(value="cs.LG")),
            # NOTE: range is for integers. so the below will not work
            FieldCondition(key="year", range=Range(gte='20241129'))
        ]
    ),
    limit=5,
    with_payload=True
)

for hit in results.points:
    payload = hit.payload
    print(f"  {payload['category']:8} {payload['year']} | {hit.score:.4f} | {payload['title'][:60]}")

Scenario 4: Combined Filter (cs.LG AND year >= 2025)
  cs.LG    20251114 | 1.0000 | Optimizing Mixture of Block Attention
  cs.LG    20251108 | 0.5668 | MoSKA: Mixture of Shared KV Attention for Efficient Long-Seq
  cs.LG    20251112 | 0.5385 | Mixture-of-Channels: Exploiting Sparse FFNs for Efficient LL
  cs.LG    20251112 | 0.4535 | Making Every Head Count: Sparse Attention Without the Speed-
  cs.LG    20251107 | 0.4523 | Attention and Compression is all you need for Controllably E


In [53]:
### Get a query vector
results = client.scroll(
    collection_name=collection_name,
    scroll_filter=Filter(
        must=[FieldCondition(key="category", match=MatchValue(value="cs.LG"))]
    ),
    limit=1,
    with_vectors=True
)
query_vector = results[0][0].vector

In [54]:
def benchmark_query(query_filter, name, iterations=100):
    """Run a query multiple times and measure average latency"""
    # Warmup
    for _ in range(5):
        client.query_points(
            collection_name=collection_name,
            query=query_vector,
            query_filter=query_filter,
            limit=10,
            with_payload=True
        )

    # Actual measurement
    times = []
    for _ in range(iterations):
        start = time.time()
        client.query_points(
            collection_name=collection_name,
            query=query_vector,
            query_filter=query_filter,
            limit=10,
            with_payload=True
        )
        times.append((time.time() - start) * 1000)  # Convert to ms

    avg_time = np.mean(times)
    std_time = np.std(times)
    return avg_time, std_time

In [55]:
print("Benchmarking Qdrant performance...")
print("=" * 80)

### Scenario 1: Unfiltered
avg, std = benchmark_query(None, "Unfiltered")
print(f"Unfiltered search:        {avg:.2f}ms (±{std:.2f}ms)")
baseline = avg

### Scenario 2: Category filter
filter_category = Filter(
    must=[FieldCondition(key="category", match=MatchValue(value="cs.LG"))]
)
avg, std = benchmark_query(filter_category, "Category filter")
overhead = avg / baseline
print(f"Category filter:          {avg:.2f}ms (±{std:.2f}ms) | {overhead:.2f}x overhead")

### Scenario 3: Year filter
filter_year = Filter(
    must=[FieldCondition(key="year", range=Range(gte=2025))]
)
avg, std = benchmark_query(filter_year, "Year filter")
overhead = avg / baseline
print(f"Year filter (>= 2025):    {avg:.2f}ms (±{std:.2f}ms) | {overhead:.2f}x overhead")

### Scenario 4: Combined filters
filter_combined = Filter(
    must=[
        FieldCondition(key="category", match=MatchValue(value="cs.LG")),
        FieldCondition(key="year", range=Range(gte=2025))
    ]
)
avg, std = benchmark_query(filter_combined, "Combined filter")
overhead = avg / baseline
print(f"Combined filter:          {avg:.2f}ms (±{std:.2f}ms) | {overhead:.2f}x overhead")

print("=" * 80)

Benchmarking Qdrant performance...
Unfiltered search:        8.05ms (±1.31ms)
Category filter:          9.12ms (±0.88ms) | 1.13x overhead
Year filter (>= 2025):    8.69ms (±7.26ms) | 1.08x overhead
Combined filter:          8.76ms (±1.32ms) | 1.09x overhead


In [ ]:
# NOTE: weird that they don't require an API to access their servers???

In [ ]:
# PINECONE;
!PINECONE_API_KEY="your-api-key-here"

from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv

### Load API key
load_dotenv()
api_key = os.getenv('PINECONE_API_KEY')

In [ ]:
### Initialize Pinecone
pc = Pinecone(api_key=api_key)

### Create index
index_name = "arxiv-papers-5k"

### Delete index if it exists
if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)
    print(f"Deleted existing index: {index_name}")
    time.sleep(5)  # Wait for deletion to complete

### Create new index
pc.create_index(
    name=index_name,
    dimension=1536,  # Cohere embedding dimension
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"  # Free tier only supports us-east-1
    )
)
print(f"Created index: {index_name}")

### Wait for index to be ready
while not pc.describe_index(index_name).status['ready']:
    print("Waiting for index to be ready...")
    time.sleep(1)

### Connect to index
index = pc.Index(index_name)

In [ ]:
print(f"\nLoading {len(papers_df)} papers into Pinecone...")

### Prepare vectors for upload
### Pinecone expects: (id, vector, metadata)
vectors = []
for idx, row in papers_df.iterrows():
    # Truncate authors field to avoid hitting metadata size limits
    # Pinecone has a 40KB metadata limit per vector
    authors = row['authors'][:500] if len(row['authors']) > 500 else row['authors']

    vector = {
        "id": row['id'],
        "values": embeddings[idx].tolist(),
        "metadata": {
            "title": row['title'],
            "authors": authors,
            "abstract": row['abstract'],
            "year": int(row['year']),
            "category": row['categories']
        }
    }
    vectors.append(vector)

    # Upload in batches of 100
    if len(vectors) == 100:
        index.upsert(vectors=vectors)
        print(f"  Uploaded {idx + 1} / {len(papers_df)} papers")
        vectors = []

### Upload remaining vectors
if vectors:
    index.upsert(vectors=vectors)
    print(f"  Uploaded {len(papers_df)} / {len(papers_df)} papers")

print("\nUpload complete!")

### Pinecone has eventual consistency
### Wait a bit for all vectors to be indexed
print("Waiting for indexing to complete...")
time.sleep(10)

### Verify
stats = index.describe_index_stats()
print(f"\nIndex '{index_name}' now has {stats['total_vector_count']} vectors")

In [ ]:
### Get a query vector from a machine learning paper
results = index.query(
    vector=[0] * 1536,  # Dummy vector just to use filter
    filter={"category": {"$eq": "cs.LG"}},
    top_k=1,
    include_metadata=True,
    include_values=True
)

query_match = results['matches'][0]
query_vector = query_match['values']
query_metadata = query_match['metadata']

print("Query paper:")
print(f"  Title: {query_metadata['title']}")
print(f"  Category: {query_metadata['category']}")
print(f"  Year: {query_metadata['year']}")
print()

### Scenario 1: Unfiltered similarity search
print("=" * 80)
print("Scenario 1: Unfiltered Similarity Search")
print("=" * 80)

results = index.query(
    vector=query_vector,
    top_k=6,  # Get 6 so we can skip the query paper itself
    include_metadata=True
)

for match in results['matches'][1:6]:  # Skip first result (query paper)
    metadata = match['metadata']
    print(f"  {metadata['category']:8} {metadata['year']} | {match['score']:.4f} | {metadata['title'][:60]}")
print()

### Scenario 2: Filter by category
print("=" * 80)
print("Scenario 2: Category Filter (cs.LG only)")
print("=" * 80)

results = index.query(
    vector=query_vector,
    filter={"category": {"$eq": "cs.LG"}},
    top_k=6,
    include_metadata=True
)

for match in results['matches'][1:6]:
    metadata = match['metadata']
    print(f"  {metadata['category']:8} {metadata['year']} | {match['score']:.4f} | {metadata['title'][:60]}")
print()

### Scenario 3: Filter by year range
print("=" * 80)
print("Scenario 3: Year Filter (2025 or later)")
print("=" * 80)

results = index.query(
    vector=query_vector,
    filter={"year": {"$gte": 2025}},
    top_k=5,
    include_metadata=True
)

for match in results['matches']:
    metadata = match['metadata']
    print(f"  {metadata['category']:8} {metadata['year']} | {match['score']:.4f} | {metadata['title'][:60]}")
print()

### Scenario 4: Combined filters
print("=" * 80)
print("Scenario 4: Combined Filter (cs.LG AND year >= 2025)")
print("=" * 80)

results = index.query(
    vector=query_vector,
    filter={
        "$and": [
            {"category": {"$eq": "cs.LG"}},
            {"year": {"$gte": 2025}}
        ]
    },
    top_k=5,
    include_metadata=True
)

for match in results['matches']:
    metadata = match['metadata']
    print(f"  {metadata['category']:8} {metadata['year']} | {match['score']:.4f} | {metadata['title'][:60]}")

In [ ]:
### Get a query vector
results = index.query(
    vector=[0] * 1536,
    filter={"category": {"$eq": "cs.LG"}},
    top_k=1,
    include_values=True
)
query_vector = results['matches'][0]['values']

def benchmark_query(query_filter, name, iterations=100):
    """Run a query multiple times and measure average latency"""
    # Warmup
    for _ in range(5):
        index.query(
            vector=query_vector,
            filter=query_filter,
            top_k=10,
            include_metadata=True
        )

    # Actual measurement
    times = []
    for _ in range(iterations):
        start = time.time()
        index.query(
            vector=query_vector,
            filter=query_filter,
            top_k=10,
            include_metadata=True
        )
        times.append((time.time() - start) * 1000)  # Convert to ms

    avg_time = np.mean(times)
    std_time = np.std(times)
    return avg_time, std_time

print("Benchmarking Pinecone performance...")
print("=" * 80)

### Scenario 1: Unfiltered
avg, std = benchmark_query(None, "Unfiltered")
print(f"Unfiltered search:        {avg:.2f}ms (±{std:.2f}ms)")
baseline = avg

### Scenario 2: Category filter
avg, std = benchmark_query({"category": {"$eq": "cs.LG"}}, "Category filter")
overhead = avg / baseline
print(f"Category filter:          {avg:.2f}ms (±{std:.2f}ms) | {overhead:.2f}x overhead")

### Scenario 3: Year filter
avg, std = benchmark_query({"year": {"$gte": 2025}}, "Year filter")
overhead = avg / baseline
print(f"Year filter (>= 2025):    {avg:.2f}ms (±{std:.2f}ms) | {overhead:.2f}x overhead")

### Scenario 4: Combined filters
avg, std = benchmark_query(
    {"$and": [{"category": {"$eq": "cs.LG"}}, {"year": {"$gte": 2025}}]},
    "Combined filter"
)
overhead = avg / baseline
print(f"Combined filter:          {avg:.2f}ms (±{std:.2f}ms) | {overhead:.2f}x overhead")

print("=" * 80)

In [ ]:
"""
Choose pgvector when:
    - Raw query speed is critical (need sub-5ms)

    - You already have PostgreSQL infrastructure

    - Your team has strong SQL and PostgreSQL skills

    - You primarily filter on integer fields (dates, IDs, counts)

    - Your scale is moderate (up to a few million vectors)

    - You’re comfortable with PostgreSQL operational tasks (VACUUM, index maintenance)

Choose Qdrant when:
    - in need of predictable performance regardless of filter complexity

    - filter on many fields with unpredictable combinations

    - You can accept ~50ms baseline latency

    - You want self-hosting but need better filtering than ChromaDB

    - You’re comfortable with Docker or Kubernetes deployment

    - Your scale is millions to tens of millions of vectors

Choose Pinecone when:
    - want zero operational overhead

    - team should focus on features, not database operations

    - can accept ~100ms baseline latency (varies by geography)

    - need heavy filtering on multiple metadata fields

    - want automatic scaling without capacity planning

    - scale could grow unpredictably

Choose ChromaDB when:
    - prototyping and learning

    - need simple local development

    - Filtering is occasional, not critical path

    - want minimal setup complexity

    - scale is small (thousands to tens of thousands of vectors)
"""

'\nChoose pgvector when:\n    - Raw query speed is critical (need sub-5ms)\n\n    - You already have PostgreSQL infrastructure\n\n    - Your team has strong SQL and PostgreSQL skills\n\n    - You primarily filter on integer fields (dates, IDs, counts)\n\n    - Your scale is moderate (up to a few million vectors)\n\n    - You’re comfortable with PostgreSQL operational tasks (VACUUM, index maintenance)\n\nChoose Qdrant when:\n    - in need of predictable performance regardless of filter complexity\n    - filter on many fields with unpredictable combinations\n    You can accept ~50ms baseline latency\n    You want self-hosting but need better filtering than ChromaDB\n    You’re comfortable with Docker or Kubernetes deployment\n    Your scale is millions to tens of millions of vectors\nChoose Pinecone when:\n\nYou want zero operational overhead\nYour team should focus on features, not database operations\nYou can accept ~100ms baseline latency (varies by geography)\nYou need heavy filterin

In [ ]:
"""
All four databases use approximate nearest-neighbor indexes for speed.
That means queries are fast, but they can sometimes miss the true closest
results—especially when filters are applied. In real applications,
one should measure not just latency, but also result ***quality (recall)***,
and tune settings if needed.
"""

In [ ]:
"""
When building an application:

- Start with requirements. What are the latency needs? How complex are the filters? What scale is being targeting?

- Match requirements to database characteristics.
  Need speed? Consider pgvector. Need consistent filtering?
  Look at Qdrant. Want zero ops? Try Pinecone.

- Prototype quickly. Spin up a test with the actual data and query patterns.
  Measure what matters for the use case.

- Be ready to change. Your requirements might evolve.
  The database that works at 10,000 vectors might not work at 10 million.
  That’s fine. Migration can be done.
"""

In [ ]:
"""
Key Takeaways;
    Performance Patterns Are Clear: pgvector delivers 2.5ms baseline (fastest),
    Qdrant 52ms (moderate with HTTP overhead), Pinecone 87ms (network latency dominates).
    Each optimizes for different goals.

    Filtering Overhead Varies Dramatically: ChromaDB shows 3-8x overhead.
    pgvector shows 2.3x for text but 1.0x for integers. Qdrant maintains
    consistent 1.1x regardless of filter complexity.
    Pinecone achieves essentially zero filtering overhead (1.0x).

    Three Distinct Strategies Emerge: Optimize for raw speed (pgvector),
    optimize for filtering consistency (Qdrant), or optimize for convenience (Pinecone).
    No universal "best" choice exists.

    Purpose-Built Databases Excel at Filtering: Qdrant and Pinecone,
    designed specifically for filtered vector search, handle complex filters
    without performance degradation. pgvector leverages PostgreSQL's
    strengths but wasn't built primarily for this use case.

    Operational Overhead Is Real: pgvector requires PostgreSQL expertise
    (VACUUM, index maintenance). Qdrant needs container orchestration.
    Pinecone removes ops but introduces vendor dependency.
    Match operational capacity to database choice.

    Geography Matters for Cloud Services: Pinecone's 87ms baseline
    from Mexico City to AWS us-east-1 is dominated by network latency.
    Self-hosted options (pgvector, Qdrant) don't have this variance.

    Scale Changes Everything: We tested 5,000 vectors. Behavior at 50k, 500k,
    or 5M vectors will differ. The patterns we observed likely hold,
    but absolute numbers will change. Always benchmark at your target scale.

    Decision Frameworks Beat Feature Lists: Choose based on your
    constraints: latency requirements, filter complexity, existing infrastructure,
    team expertise, and operational capacity. Not on marketing claims.

    Prototyping Beats Speculation: The fastest way to know if a database works
    for you is to load your actual data and run your actual queries.
    Benchmarks guide thinking but can't replace hands-on testing.
"""